# 03 · Línea base asistida y evaluador común

**Solo etapa 3. Referencia asistida por etiquetas, no desplegable.** SAM 2.1 tiny totalmente congelado. No DINOv3, entrenamiento, disección ni demo. Toda inferencia se ejecuta exclusivamente por el usuario en **Google Colab con GPU**; no ejecutar modelos en la PC, tampoco en CPU.

Prerrequisitos: etapas 1–2 verificadas y la misma raíz Drive con `raw/`, `stage2/`, `regimes/*/manifests/` y `preprocessing/`. Se conservan la consulta de licencia pendiente y las evidencias históricas. Este notebook no adquiere datos, no reconstruye splits y no recalcula estadísticas. [Instrucciones y evidencia requerida](../docs/project/validacion_etapa3.md).


## 1. Verificar el paquete y reutilizar el entorno

En la PC ejecutar `python scripts/export_stage3_bundle.py`. Subir `artifacts/auto-prompt-sam2-stage3.zip` a Drive, abrir este notebook en Colab, seleccionar GPU y copiar el SHA-256 mostrado en la celda. Usar un directorio nuevo de fuentes; conservar los ZIP históricos. La instalación reutiliza el bootstrap y pines de etapas 1–2 sin ampliarlos.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive

drive.mount('/content/drive')
PERSIST_ROOT = '/content/drive/MyDrive/auto-prompt-sam2'
REPO = Path('/content/auto-prompt-sam2-stage3')
PYTHON = Path('/content/auto-prompt-venv/bin/python')
BUNDLE = Path('/content/drive/MyDrive/auto-prompt-sam2-stage3.zip')
EXPECTED_BUNDLE_SHA256 = ''  # Copiar salida de export_stage3_bundle.py.
assert len(EXPECTED_BUNDLE_SHA256) == 64, 'Configura el SHA-256 del paquete.'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_BUNDLE_SHA256
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        assert (REPO / item.filename).resolve().is_relative_to(REPO.resolve())
    archive.extractall(REPO)
child_env = dict(os.environ, PYTHONPATH=str(REPO / 'src'),
    AUTO_PROMPT_PERSIST_ROOT=PERSIST_ROOT, PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8',
    AUTO_PROMPT_EXECUTION_CONTEXT='google_colab_user_notebook')
# Token opcional para descarga autorizada, nunca imprimirlo ni incluirlo en informes.
# from google.colab import userdata
# child_env['HF_TOKEN'] = userdata.get('HF_TOKEN')
LOG_DIR = Path(PERSIST_ROOT) / 'stage3/notebook_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
def command(args, log_name):
    with (LOG_DIR / log_name).open('w', encoding='utf-8') as log:
        process = subprocess.Popen(args, cwd=REPO, env=child_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        result = process.wait()
    if result:
        raise RuntimeError(f'Proceso terminado con código {result}; conservar salida y log {log_name}')
if not PYTHON.is_file():
    command([sys.executable, str(REPO / 'scripts/bootstrap_colab.py')], 'bootstrap.log')
command([str(PYTHON), '-m', 'pip', 'install', '--no-deps', '-r', str(REPO / 'requirements-stage2.txt')], 'stage2_pins.log')
command([str(PYTHON), '-m', 'pip', 'check'], 'pip_check.log')


## 2. Protocolo fijado antes de ejecutar

`configs/baseline.yaml`: semilla 42; conectividad 8; máximo 1 positivo en agua válida y 1 negativo en no agua válida **por componente**; separación mínima 8 px dentro de cada llamada. Permutación PCG64 sin reemplazo, semilla por régimen/ID, componentes ordenados por primer píxel (fila,columna), desempates por orden. No se eliminan componentes pequeños ni se relaja la distancia; se registran faltantes. La ventaja de conocer componentes reales debe declararse al comparar con generadores automáticos futuros; estos deberán alinear presupuesto y geometría en su etapa.

Cada componente recibe una llamada multimáscara, se elige el mayor score de IoU **predicha** (primer índice en empates) y se unen máscaras binarias en 512×512. El adaptador solo recibe imagen y prompts. La imagen usa `sam_predictor_image` y el wrapper oficial aplica resize 1024 y normalización una vez; las coordenadas de entrada son `(x,y)` nativas. No hay filtrado por etiquetas de la máscara inferida.

**Sin agua real válida:** máscara vacía sin llamar a SAM; esto aprovecha el conocimiento de ausencia de agua de las etiquetas y no representa detección automática. Los chips totalmente inválidos tampoco llaman a SAM. Los negativos nunca caen en agua de otro componente ni en `-1`.

Se procesa el primer chip `train` por ID como comprobación y **toda `val`** (89 official, 75 leave_paraguay_out). Las métricas train y val permanecen separadas. Test, Bolivia y Paraguay reservado no se abren. Configuración común prefijada: no ajustar la segunda rama después de observar la primera.

No existe función de pérdida en esta etapa sin entrenamiento. El evaluador ignora `-1`; chips sin agua se excluyen de IoU individual aun con FP, se mantienen en agregados y reportan FP en píxeles/m². Área desde georreferenciación EPSG:4326→EPSG:6933; no grados cuadrados ni 100 m² nominales. Cocientes nulos: `null` con motivo. mIoU: media de IoU de clases definidas con exclusiones explícitas. Por evento: acumular confusiones y luego promediar eventos con igual peso; por chip: mediana y cuartiles lineales.


## 3. Preparar baseline de desarrollo en GPU

Este comando ejecuta pruebas CPU sintéticas, comprueba pines/GPU, reutiliza generaciones de etapa 2, descarga **solo SAM** en la revisión fijada a Drive y registra su SHA-256. Congela todo, usa eval/float32 sin autocast ni posprocesamiento adicional. Guarda snapshot del código actual, configuración, hardware, puntos, faltantes, scores, máscaras, confusiones, métricas y un índice con hashes. Comprueba repetición de un chip con prompts por régimen y hash del estado completo antes/después.

`prepare` crea un run nuevo. Para continuar uno interrumpido, detener primero el escritor anterior y usar `resume` con el **mismo ZIP/configuración**. No lanzar dos escritores. Si falta una generación compatible de etapa 2, detenerse y revisar la raíz persistente; no recalcularla aquí.


In [ ]:
MODE = 'prepare'  # 'resume' solo para el mismo run interrumpido y escritor anterior detenido.
assert MODE in ('prepare', 'resume')
command([str(PYTHON), str(REPO / 'scripts/run_stage3.py'), MODE,
         '--root', PERSIST_ROOT], MODE + '.log')


## 4. Recuperación en runtime nuevo

Guardar las salidas y el ZIP indicado por el comando anterior. **Desconectar y eliminar el runtime de Colab**, volver a conectar con GPU y ejecutar solo la sección 1 y la siguiente celda, con el mismo ZIP, hash y Drive. No repetir `prepare`: crearía otro run.

`recover` lee el ticket del run compatible, carga todas las predicciones persistidas sin regenerar faltantes, compara índices/resúmenes, recarga SAM desde Drive **offline** y verifica el hash del checkpoint y estado. Regenera puntos y repite un chip con agua de cada régimen. Tolerancias prefijadas: 0 píxeles distintos, scores `atol=rtol=1e-5`, mismo índice seleccionado. Se guardan PID/boot ID para distinguir proceso nuevo de runtime nuevo; documentar cualquier cambio de GPU. Un fallo no autoriza a cambiar tolerancias para hacerlo pasar.


In [ ]:
command([str(PYTHON), str(REPO / 'scripts/run_stage3.py'), 'recover',
         '--root', PERSIST_ROOT], 'recover.log')


## 5. Compartir evidencia y detenerse

Compartir **este notebook ejecutado** (Archivo → Descargar → .ipynb, con celdas/salidas/errores), el ZIP del run que imprime `recover`, y `validation.json` / `recovery_validation.json` dentro de `PERSIST_ROOT/stage3/evidence/<run_id>/`. El ZIP contiene configuración, entorno, snapshot, marcadores, puntos, scores, máscaras NPZ y confusiones de ambos regímenes; excluye datos originales y pesos. Conservar logs de `stage3/notebook_logs/` si ocurre un error.

El asistente revisará evidencia, hashes, roles, congelamiento, repetición y recarga antes de cerrar. **No presentar métricas de los stubs como SAM real ni marcar la etapa concluida por crear este notebook.** No continuar a etapa 4, DINOv3, entrenamiento o demo. IA: Codex preparó código, contratos, pruebas y documentación; el usuario ejecuta Colab y el equipo revisa las decisiones científicas.
